# Notebook 02: Cleaning and Feature Engineering

## Overview

Our second notebook takes the disaster-level dataset produced by Notebook 01 and prepares it for modeling. The main tasks here are: handling missing values, standardizing the incident type categories, engineering time-based and geographic features, adjusting dollar amounts for inflation, and creating the classification target variable.

All engineering decisions we took here are guided by the idea that features must only use information that is available at the exact moment a Presidential disaster declaration is signed to avoid data leakage and reproduce realistic scenario, so that the modle can create value for the real-world application as of Day-1 business decisions.

In [1]:
import pandas as pd
import numpy as np
import sys
#parent dir contains utils.py with shared helpers
sys.path.append('../')
from utils import get_season, add_prior_disasters, data_summary, DISASTER_BINS

PROCESSED = '../data/processed/'
#load disaster-level output from nb01
disas = pd.read_csv(PROCESSED + 'merged_disaster_level.csv', low_memory=False)
print('Disaster-level:', disas.shape)

Disaster-level: (1766, 14)


## 2.1 Column Review and Leakage Check

Our first step is to confirm that no post-declaration information is present in the feature set. In this dataset, "total_federal_share" is the only financial column, and it is used exclusively to construct the classification label. It is never passed to the model as an input feature. The remaining columns are either event metadata (incident type, state, dates) or county-level contextual information (population, income, poverty, risk score), all of which are knowable at the time of declaration. Therefore, no columns need to be dropped for leakage reasons.

In [2]:
#inspect column names + null counts -> confirm no leakage columns are here
print('Disaster-level columns:')
print(list(disas.columns))
data_summary(disas, 'Disaster-Level Raw')

Disaster-level columns:
['disasterNumber', 'total_federal_share', 'n_projects', 'n_counties', 'incidentType', 'stateCode', 'stateAbbreviation', 'declarationDate', 'incidentBeginDate', 'incidentEndDate', 'population', 'median_income', 'poverty_rate', 'risk_score']

  Disaster-Level Raw  |  1,766 rows  x  14 cols
Columns with nulls:
  median_income                             67  (3.8%)
  poverty_rate                              67  (3.8%)
  risk_score                               110  (6.2%)



## 2.2 Parse Date Columns

The date columns arrive as strings from the CSV. We converted them to datetime objects before any time-based feature engineering is done. Parsing errors are coerced to null rather than raising exceptions, as a small number of records may have malformed date strings.

In [3]:
DATE_COLS = ['declarationDate', 'incidentBeginDate', 'incidentEndDate']
#convert string -> datetime, errors='coerce' turns bad strings into nat
for col in DATE_COLS:
    if col in disas.columns:
        disas[col] = pd.to_datetime(disas[col], errors='coerce')
print('Date columns parsed.')

Date columns parsed.


## 2.3 Handle Missing Values

Missing values in the socioeconomic features ("median_income", "poverty_rate", "population", "risk_score") affect approximately 3 to 6 percent of records. These arise because some county FIPS codes in the FEMA data correspond to territories or administrative units not covered by the 2019 ACS or NRI files.

Median imputation is used rather than mean imputation because the distributions of these variables are right-skewed. Using the median preserves the central tendency without allowing a small number of high-income or high-population outliers to inflate imputed values. The imputation is applied to the full dataset before splitting, which is acceptable because these are static county characteristics rather than outcomes derived from the target.

Rows missing a dollar amount or an incident begin date are dropped, as neither can be imputed meaningfully.

## 2.3 Handle Missing Values

Missing values in the socioeconomic features ("median_income", "poverty_rate", "population", "risk_score") here occur because some county FIPS codes in the FEMA data correspond to territories or administrative units not covered by the 2019 ACS or NRI files.

We used median imputation rather than mean imputation as they are static baseline characteristics of a county, ans so that  we don't let a small number of high-income or high-population outliers inflate the imputed values. Instead, rows missing a dollar amount or an incident begin date are dropped entirely, as neither can be imputed meaningfully without compromising the integrity of the label or time-series logic.

In [4]:
#median imputation for county features -> handles ~3-6% missing from territory fips
for col in ['median_income', 'poverty_rate', 'population', 'risk_score']:
    if col in disas.columns:
        disas[col] = disas[col].fillna(disas[col].median())

#drop rows with no dollar amount or no date -> cannot impute these
disas = disas.dropna(subset=['total_federal_share', 'incidentBeginDate'])
print(f'After null handling -> Disaster: {disas.shape}')
data_summary(disas, 'Disaster-Level')

After null handling -> Disaster: (1766, 14)

  Disaster-Level  |  1,766 rows  x  14 cols
  No nulls  ✓



## 2.3b Incident Type Mapping

The raw FEMA dataset accumulates 25 years of inconsistent data entry. A single event category such as "Severe Storm" appears under at least six different spellings including "Severe Storm(s)", "Severe Storms", and "Thunderstorm Winds". If these variants are passed directly to a one-hot encoder, the model treats each spelling as a separate category and learns nothing generalizable.

A hand-coded lookup dictionary maps all observed variants to one of eight canonical categories: Flood, Hurricane, Severe Storm, Winter Storm, Tornado, Wildfire, Earthquake, and Other. The mapping is derived from reviewing the full distribution of raw values and grouping them by meteorological type. Any unmapped value defaults to "Other" to handle future unseen variants at inference time.

## 2.3b Incident Type Mapping

Looking at the dataset, we observed thet it has quite inconsistent data entry. A single event category such as "Severe Storm" appears under at least six different spellings including "Severe Storm(s)", "Severe Storms", and "Thunderstorm Winds". If these variants are passed directly to a one-hot encoder, the model treats each spelling as a separate category and fails to learn generalizable patterns. For this reason, we created a lookup dictionary that maps all observed variants to one of eight main categories: Flood, Hurricane, Severe Storm, Winter Storm, Tornado, Wildfire, Earthquake, and Other. We chose these after reviewing the full distribution of raw values and grouping them by meteorological type. Any unmapped value defaults to "Other".

In [5]:
#lookup dict: raw fema spelling -> canonical category
#.fillna('Other') handles any unseen variants at inference time
INCIDENT_TYPE_CANONICAL = {
    'Flood':                  'Flood',
    'Flooding':               'Flood',
    'Dam/Levee Break':        'Flood',
    'Hurricane':              'Hurricane',
    'Typhoon':                'Hurricane',
    'Tropical Storm':         'Hurricane',
    'Coastal Storm':          'Hurricane',
    'Severe Storm(s)':        'Severe Storm',
    'Severe Storms':          'Severe Storm',
    'Severe Storm':           'Severe Storm',
    'Thunderstorms':          'Severe Storm',
    'Thunderstorm Winds':     'Severe Storm',
    'Snow':                   'Winter Storm',
    'Snowstorm':              'Winter Storm',
    'Blizzard':               'Winter Storm',
    'Ice Storm':              'Winter Storm',
    'Severe Ice Storm':       'Winter Storm',
    'Freezing':               'Winter Storm',
    'Winter Storm':           'Winter Storm',
    'Winter Storms':          'Winter Storm',
    'Cold Wave':              'Winter Storm',
    'Tornado':                'Tornado',
    'Tornadoes':              'Tornado',
    'Fire':                   'Wildfire',
    'Wildfire':               'Wildfire',
    'Forest Fires':           'Wildfire',
    'Earthquake':             'Earthquake',
    'Biological':             'Other',
    'Chemical':               'Other',
    'Toxic Substances':       'Other',
    'Terrorist':              'Other',
    'Human Cause':            'Other',
    'Drought':                'Other',
    'Mud/Landslide':          'Other',
    'Landslide':              'Other',
    'Volcanic Eruption':      'Other',
    'Tsunami':                'Other',
    'Other':                  'Other',
}

before = disas['incidentType'].nunique()
#map raw -> unmapped values become 'Other'
disas['incidentType'] = disas['incidentType'].map(INCIDENT_TYPE_CANONICAL).fillna('Other')
after  = disas['incidentType'].nunique()
print(f'incidentType: {before} variants -> {after} canonical categories')
print(disas['incidentType'].value_counts())

incidentType: 26 variants -> 8 canonical categories
incidentType
Severe Storm    850
Hurricane       302
Flood           189
Winter Storm    185
Other           135
Wildfire         56
Tornado          35
Earthquake       14
Name: count, dtype: int64


## 2.4 Feature Engineering: time-based features

We created 4 time-derived features from the date columns:

* "declaration_lag_days": number of days between the incident start and the Presidential declaration. A shorter lag may indicate a more severe and politically salient event. A longer lag may reflect administrative delays or smaller-scale incidents that took time to escalate.
* "incident_duration_days": number of days from incident start to end. Longer events such as flooding or drought tend to accumulate more PA project applications than short, intense events such as tornadoes.
* "incident_season": meteorological season of the incident begin date, derived using the "get_season" helper. Season captures the climatic context: hurricane season and winter storm season produce systematically different damage profiles.
* "incident_year": only for the time-based train/validation/test split in Notebook 04. It is not passed as a model feature because leaking the year into the model could allow it to learn spurious trends in federal spending policy rather than genuine physical disaster characteristics.

In [6]:
#days from incident start -> declaration (proxy for severity + political urgency)
disas['declaration_lag_days']   = (disas['declarationDate']  - disas['incidentBeginDate']).dt.days.clip(lower=0)
#days from incident start -> end (longer events accumulate more projects)
disas['incident_duration_days'] = (disas['incidentEndDate']  - disas['incidentBeginDate']).dt.days.clip(lower=0)
#clip(lower=0) removes negative values from data entry errors
disas['incident_season']        = disas['incidentBeginDate'].dt.month.apply(get_season)
#incident_year -> used for temporal split only, not a model feature
disas['incident_year']          = disas['incidentBeginDate'].dt.year
print('Time features added.')
disas[['declaration_lag_days', 'incident_duration_days', 'incident_season', 'incident_year']].head(3)

Time features added.


,declaration_lag_days,incident_duration_days,incident_season,incident_year
0,4,9,Summer,1998
1,4,29,Fall,1998
2,23,6,Winter,1998


## 2.5 Feature Engineering: Prior Disaster Count

`prior_disasters_5yr` counts the number of federally declared PA disasters in the same state in the five years preceding each event. This feature captures two complementary signals.

On one hand, a state with a high prior count may have better-prepared emergency management infrastructure, potentially leading to faster and more efficient recovery that keeps costs lower. On the other hand, repeat disaster exposure may indicate a chronically high-risk environment where cumulative damage from prior events compounds costs.

The five-year window is a deliberate choice. A shorter window would miss multi-year drought or flood cycles. A longer window would include historical events that are less relevant to current infrastructure capacity. The `add_prior_disasters` helper in `utils.py` implements a vectorised look-back on sorted data to avoid any forward-looking bias.

## 2.5 Feature Engineering: Prior Disaster Count

The "prior_disasters_5yr" variable counts the number of federally declared PA disasters in the same state in the five years preceding each event. This feature captures two complementary signals. We considered two possible relations to this:
- On one hand, a state with a high prior count may have better-prepared emergency management infrastructure, potentially leading to faster and more efficient recovery that keeps costs lower.
- On, repeat disaster exposure may indicate a chronically high-risk environment where cumulative damage from prior events compounds costs.

We chose the 5 year window as a shorter window would miss multi-year drought or flood cycles. A longer window would include historical events that are less relevant to current infrastructure capacity.

In [7]:
#count pa declarations in same state within 5yr window before each disaster
#uses sorted date-based lookback -> no forward leakage
disas = add_prior_disasters(
    disas,
    state_col='stateAbbreviation',
    date_col='incidentBeginDate',
    window_years=5
)
print('Prior disaster counts added.')
disas[['disasterNumber', 'stateAbbreviation', 'incidentBeginDate', 'prior_disasters_5yr']].head(5)

Prior disaster counts added.


,disasterNumber,stateAbbreviation,incidentBeginDate,prior_disasters_5yr
0,1239,TX,1998-08-22 00:00:00+00:00,0
8,1268,WY,1998-10-05 00:00:00+00:00,0
1,1257,TX,1998-10-17 00:00:00+00:00,1
5,1264,LA,1998-12-22 00:00:00+00:00,0
2,1260,TN,1998-12-23 00:00:00+00:00,0


## 2.5b CPI Inflation Adjustment

The dataset spans 1998 to 2026, a period during which cumulative inflation has been approximately 100 percent. A disaster costing $40 million in 1998 represents a substantially larger real expenditure than $40 million in 2022. If tier thresholds are applied to nominal dollars, early-year events are systematically under-classified because the same real-dollar damage appears smaller in nominal terms.

To correct for this, all federal share amounts are converted to 2019 dollars using annual CPI-U averages from the Bureau of Labor Statistics. The year 2019 is chosen as the base because it is the last pre-pandemic year, avoiding the distortions introduced by COVID-era supply chain effects on construction costs. The adjustment formula is:

```
adjusted_amount = nominal_amount * (CPI_2019 / CPI_incident_year)
```

The adjusted amount is used to create the tier label in section 2.6 and is never passed as a model feature.

## 2.5b CPI Inflation Adjustment

The dataset spans 1998 to 2026. We noticed that is tier thresholds are applied directly to nominal dollars, then early-year events are systematically under-classified because the same real-dollar damage appears smaller in nominal terms. (Aka: a disaster costing $40 million in 1998 represents a substantially larger real expenditure than $40 million in 2022). For this reason we converted all federal share amounts to 2019 dollars using annual CPI-U averages from the Bureau of Labor Statistics as of 2019 (because it's the last pre-pandemic year, to avoid the distortions introduced by COVID). 

The adjusted amount is used exclusively to create the tier label in the following section and is never passed as a model feature.

In [8]:
#bls cpi-u annual averages, base 1982-84=100
CPI_BY_YEAR = {
    1998: 163.0, 1999: 166.6, 2000: 172.2, 2001: 177.1, 2002: 179.9,
    2003: 184.0, 2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.3,
    2008: 215.3, 2009: 214.5, 2010: 218.1, 2011: 224.9, 2012: 229.6,
    2013: 233.0, 2014: 236.7, 2015: 237.0, 2016: 240.0, 2017: 245.1,
    2018: 251.1, 2019: 255.7, 2020: 258.8, 2021: 271.0, 2022: 292.7,
    2023: 304.7, 2024: 314.2, 2025: 322.0, 2026: 329.0,
}
#2019 is the base year -> pre-pandemic, stable reference point
CPI_BASE = CPI_BY_YEAR[2019]

#map incident year -> cpi factor, fallback to 2019 for any unmapped years
disas['cpi_factor'] = disas['incident_year'].map(CPI_BY_YEAR).fillna(CPI_BASE)
#multiply by ratio -> converts nominal $ to 2019 $
disas['total_federal_share_2019'] = disas['total_federal_share'] * (CPI_BASE / disas['cpi_factor'])
#drop intermediate factor column -> not needed downstream
disas.drop(columns=['cpi_factor'], inplace=True)

print(f'CPI adjustment applied.')
print(f'  Raw median:  ${disas["total_federal_share"].median():>15,.0f}')
print(f'  2019 median: ${disas["total_federal_share_2019"].median():>15,.0f}')

CPI adjustment applied.
  Raw median:  $      8,917,274
  2019 median: $      9,963,193


## 2.6 Create Classification Target: Funding Tier

We structured our prediction target as a 4 class ordinal label derived by binning the CPI-adjusted total federal share into the following ranges, which correspond to FEMA's operational planning tiers:

* Tier 0: Below $1M (Minor)
* Tier 1: $1M to $50M (Moderate)
* Tier 2: $50M to $500M (Major)
* Tier 3: Above $500M (Catastrophic)


These thresholds are grounded in FEMA's own funding and staffing frameworks rather than being chosen arbitrarily based on data distributions. This makes the classification task practically interpretable for our business user: each tier corresponds to a different level of federal resource commitment.

We chose to still compute Tier 3 (Catastrophic) for completeness, but we chose to exclude it from the modeling scope in Notebook 04 for a specific reason: Catastrophic events are funded through Congressional Emergency Supplemental Appropriations rather than the standard Disaster Relief Fund (DRF). Training a model to predict these outlier events (other than it being impossible or they wouldn't be really outlier events) would generate false positives that unnecessarily consume DRF reserves meant for routine operations.

In [9]:
#drop records with zero or negative adjusted amount -> cancelled/reversed obligations
disas = disas[disas['total_federal_share_2019'] > 0].copy()
#bin cpi-adjusted amount into 4 tiers using policy-grounded thresholds
disas['funding_tier'] = pd.cut(
    disas['total_federal_share_2019'],
    bins=DISASTER_BINS,    #[0, 1M, 50M, 500M, inf] defined in utils.py
    labels=[0, 1, 2, 3],
    right=False
).astype(int)

dis_labels = {0: 'Minor (<$1M)', 1: 'Moderate ($1M-$50M)',
              2: 'Major ($50M-$500M)', 3: 'Catastrophic (>$500M)'}
print('Disaster-level tier distribution (CPI-adjusted 2019 $):')
for t, n in disas['funding_tier'].value_counts().sort_index().items():
    print(f'  Tier {t}  {dis_labels[t]:<30}  {n:>5,}  ({100*n/len(disas):.1f}%)')

Disaster-level tier distribution (CPI-adjusted 2019 $):
  Tier 0  Minor (<$1M)                      145  (8.2%)
  Tier 1  Moderate ($1M-$50M)             1,296  (73.4%)
  Tier 2  Major ($50M-$500M)                254  (14.4%)
  Tier 3  Catastrophic (>$500M)              71  (4.0%)


## 2.7 Final Check and Save

A final data quality check confirms no remaining nulls before saving. The cleaned file is the direct input to notebook 03 (EDA) and notebook 04 (modeling).

In [10]:
#final null check before saving -> confirms clean output for nb03 + nb04
data_summary(disas, 'Cleaned Disaster-Level')
#save to processed folder -> direct input for nb03 eda + nb04 modeling
disas.to_csv(PROCESSED + 'cleaned_disaster_level.csv', index=False)
print(f'Saved: cleaned_disaster_level.csv  ->  {disas.shape}  |  target: funding_tier (0-3)')


  Cleaned Disaster-Level  |  1,766 rows  x  21 cols
  No nulls  ✓

Saved: cleaned_disaster_level.csv  ->  (1766, 21)  |  target: funding_tier (0-3)
